In [ ]:
##############################################
##############################################
### TO GET ACCESS TO FILE IN GOOGLE COLAB ###
##############################################
##############################################

# To get access to the data files from github in Google Colab
!git clone https://github.com/math869c/graph_representation_st457.git

# set the folder to get access to the data
import os
os.chdir('/content') # to avoid error if rerun
os.chdir('./graph_representation_st457') # gives them access to everything in the folder

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import torch.optim as optim
import json
import optuna
import time

# load from other folders
from helper_functions import *
from model_classes import *

In [ ]:
# Stock prices
open_prices_interp = pd.read_csv('data_folder/open_prices_interp.csv', index_col=0)
# into numpy
x = open_prices_interp.to_numpy()
tickers_with_data = open_prices_interp.columns

# Load graph data
with open("data_folder/firm_industry.json", "r") as f:
    firm_industry_dict = json.load(f)
A = np.load("data_folder/adjacency_matrix.npy")
adj_matrix = torch.tensor(A, dtype=torch.float32)

# make training data
batch_size =32
X_train, y_train, X_val, y_val, X_test, y_test, sc, train_loader, val_loader, test_loader = create_data(x,
                                                                                                        batch_size =batch_size,
                                                                                                        flatten_data = True, # Should be True for LSTM, False for GTC and GAT
                                                                                                        flatten_time_features=False # Should be True for GAT, False for LSTM and GTC
                                                                                                        )

# building optuna objective function for hyperparameter tuning

In [ ]:
SEARCH_N_TRIALS = 30

best_model_state_global = None
best_loss_global = float("inf")

def objective(trial):
    global best_model_state_global, best_loss_global

    hidden_units = trial.suggest_int("lstm_units", 64, 256, step=64)
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 5e-4, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    epochs = trial.suggest_int("epochs", 15, 40)
    patience = trial.suggest_int("patience", 5, 10)

    model = LSTMRegressor(
        input_size=X_train.shape[2],
        hidden_units=hidden_units,
        output_size=y_train.shape[1],
        dropout_rate=dropout_rate
    )

    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    best_val_loss = float("inf")
    best_state_dict = None
    patience_counter = 0
    trial_start_time = time.time()

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch_LSTM(model, train_loader, criterion, optimizer)
        val_loss, val_acc = evaluate_LSTM(model, val_loader, criterion)

        print(
            f"Trial {trial.number + 1}/{SEARCH_N_TRIALS} | "
            f"Epoch {epoch + 1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}"
        )

        trial.report(val_loss, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = copy.deepcopy(model.state_dict())
            patience_counter = 0
            trial.set_user_attr("best_val_acc", val_acc)
            if val_loss < best_loss_global:
                best_loss_global = val_loss
                best_model_state_global = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    trial.set_user_attr("training_time_s", time.time() - trial_start_time)
    return best_val_loss


# Fine tune model

In [ ]:
# run trials
best_overall_loss, best_config, best_model_final, best_history_final = 100.0, {}, None, None

results_log = []
search_start_time = time.time()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=SEARCH_N_TRIALS)

for trial in study.trials:
    if trial.value is None:
        continue

    results_log.append({
        'config_id': trial.number + 1,
        'params': str(trial.params),
        'val_loss': trial.value,
        'training_time_s': trial.user_attrs.get('training_time_s', np.nan)
    })

results_df = pd.DataFrame(results_log).sort_values(by='val_loss', ascending=True)
search_end_time = time.time()
total_duration = search_end_time - search_start_time

best_params = study.best_trial.params
best_config = dict(best_params)
best_overall_loss = study.best_trial.value

print("\n" + "=" * 60)
print(f"Optuna Search Complete in {total_duration:.2f} seconds.")
print(f"Best Configuration: {best_config} | Best Val loss: {best_overall_loss:.4f}")

print("\n--- Detailed Hyperparameter Search Results ---")
print(results_df)

print("\nBest trial:")
print("  Value:", study.best_trial.value)
print("  Params:")
for k, v in study.best_trial.params.items():
    print(f"    {k}: {v}")


# Save best model 

In [ ]:
output_size = y_train.shape[1]
input_size = X_train.shape[2]

model_LSTM = LSTMRegressor(
    input_size=input_size,
    hidden_units=best_params['lstm_units'],
    output_size=output_size,
    dropout_rate=best_params['dropout_rate']
)
model_LSTM.load_state_dict(best_model_state_global)

criterion = nn.MSELoss()
optimizer = optim.Adam(model_LSTM.parameters(), lr=best_params['learning_rate'])
MODEL_SAVE_PATH = 'best_lstm_model_graphtheory.pt'

torch.save({
    'model_state_dict': model_LSTM.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'hyperparameters': best_params,
}, MODEL_SAVE_PATH)

In [ ]:
test_loss_LSTM, _ = evaluate_LSTM(model_LSTM, test_loader, criterion)
print("Test loss:", test_loss_LSTM)
